In [1]:
import ee
import geemap
from dotenv import load_dotenv
import os
load_dotenv(dotenv_path='.env')
project_name = os.getenv("project_name")
# print(f"Project Name: {project_name}")
ee.Initialize(project=project_name)

In [2]:
map = geemap.Map()

point = ee.Geometry.Point([90.4152, 23.8041]) #around dhaka
region = ee.Geometry.Rectangle([90.3, 23.7, 90.5, 23.9]) #bounding box around the point

top_left = [23.822424724001266, 90.46289150228976]
bottom_right = [23.796369273111445, 90.5071668854189]

#region = ee.Geometry.Rectangle([top_left[1], bottom_right[0], bottom_right[1], top_left[0]])

map.centerObject(point, 10)
map.addLayer(point, {'color': 'red'}, 'Point Layer')
map.addLayer(region, {'color': 'blue'}, 'Region Layer')

map #just testing if everything is working

Map(center=[23.8041, 90.4152], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

In [6]:
sar_image = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filterBounds(region) \
    .filterDate('2024-01-01', '2024-02-01') \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .median() \
    .clip(region)

In [7]:
#removing noise if there are noise
smoothed = sar_image.focal_median(50, 'circle', 'meters')

In [8]:
vh = smoothed.select('VH')
threshold = -18

#we are selecting vh as it is more sensitive to water bodies 
#the threshold value may vary based on location and time of year, overall it tells us how strong the signal is being reflected back to the satellite

water_mask = vh.lt(threshold).selfMask()


In [9]:
m = geemap.Map(center = region.centroid().coordinates().getInfo()[::-1], zoom = 10)

m.addLayer(smoothed, {'min': -25, 'max': -5}, 'Raw Radar Image', False) 
m.addLayer(water_mask, {'palette': ['blue']}, 'Detected Water Surface')

m

Map(center=[23.800006561617693, 90.39999999999813], controls=(WidgetControl(options=['position', 'transparent_…

In [10]:
area_image = ee.Image.pixelArea()
water_area_img = area_image.updateMask(water_mask)
stats = water_area_img.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=region,
    scale=10,
    maxPixels=1e9
)
area_sq_km = ee.Number(stats.get('area')).divide(1e6)
#we divide by 1 million to convert from square meters to square kilometers because the pixelArea function gives area in square meters
print(f"water threshold value: {threshold}")
print(f"detected water are: {area_sq_km.getInfo()} square kilometers")


water threshold value: -18
detected water are: 86.7881085783185 square kilometers
